<a href="https://colab.research.google.com/github/russelfordguinoo-oss/Russel-Repository-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/russelfordguinoo-oss/Russel-Repository-/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

I treat this as a binary classification problem because `is_declining_label` indicates whether a content item is declining.

I will start with Logistic Regression because it is a simple and interpretable classification method that produces probabilities that can also be used to rank content for review. This fits the goal of comparing a learned model with the Week-4 rule-based baseline without adding unnecessary complexity.

The Week-4 baseline used `days_since_last_update` and `impressions_90d` to rank content. I will use the same baseline logic for the comparison, but evaluate both approaches on the same held-out test set.

I will not use `trend_pct` or `trend_direction` as predictive features because they are used to derive the target label and would create target leakage. I will also exclude `content_id` and `client_id` from predictive features. `client_id` will be used only for the grouped train/test split.
```



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

The Week-4 baseline did not use a train/test split because it was a rule-based ranking over the full dataset. For Week 5, I introduce a held-out evaluation set so the learned model can be evaluated honestly.

I will split by `client_id` rather than by individual rows. This keeps content from the same client entirely within either the training or test set and reduces the risk that the model benefits from seeing the same client's patterns during training.

I will use an 80/20 grouped split with a fixed random seed for reproducibility. Both the baseline and Logistic Regression will be evaluated on exactly the same test rows.

The evaluation metric will be Precision@20 because the baseline's purpose is to produce a ranked review queue and the Week-4 review focused on the top 20 recommendations.

In [103]:
!git clone https://github.com/russelfordguinoo-oss/Russel-Repository-.git

fatal: destination path 'Russel-Repository-' already exists and is not an empty directory.


In [104]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique clients:", df["client_id"].nunique())

Rows: 30000
Columns: 44
Unique clients: 32


In [105]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


In [106]:
# Create the target label
# This is the outcome we want the model to predict.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


In [107]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

# Verify that no client appears in both sets
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients & test_clients

print("\nClient overlap:", len(overlap))
print("Overlapping clients:", overlap)

Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7

Client overlap: 0
Overlapping clients: set()


In [108]:
print("All columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

All columns:
 1. content_id
 2. client_id
 3. search_volume
 4. competition
 5. competition_level
 6. cpc
 7. content_type
 8. main_intent
 9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct
45. is_declining_label


In [109]:
# Inspect data types and missing values
feature_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique()
})

feature_audit

,dtype,missing,unique_values
content_id,object,0,30000
client_id,object,0,32
search_volume,float64,2468,41
competition,float64,2468,101
competition_level,object,2610,3
cpc,float64,2468,915
content_type,object,0,3
main_intent,object,2374,4
word_count,float64,7699,5476
char_count,float64,7699,14839


In [110]:
from pathlib import Path

dictionary_path = Path("docs/data-dictionary.md")

print(dictionary_path.exists())

True


In [111]:
print(dictionary_path.read_text()[:12000])

# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_id` | text | Pseudo

In [112]:
# Define the features for the Logistic Regression model

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

target = "is_declining_label"

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(numeric_features) + len(categorical_features))

Numeric features: 22
Categorical features: 10
Total model features: 32


In [113]:
# Verify that no forbidden columns are included

forbidden_features = {
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
}

selected_features = set(numeric_features + categorical_features)

unexpected = selected_features & forbidden_features

print("Forbidden features accidentally included:", unexpected)

Forbidden features accidentally included: set()


In [114]:
numeric_features
categorical_features
target

'is_declining_label'

In [115]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numeric features:
# - Fill missing values with the training median
# - Standardize values for Logistic Regression
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical features:
# - Treat missing categories as "unknown"
# - Convert categories into one-hot encoded columns
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Apply the appropriate preprocessing to each feature type
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [116]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

print("Logistic Regression pipeline created.")

Logistic Regression pipeline created.


In [117]:
# Separate features and target
X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target]

# Train the Logistic Regression pipeline
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [118]:
# Generate probabilities for the positive class: declining
model_probabilities = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(model_probabilities))
print("Minimum probability:", model_probabilities.min())
print("Maximum probability:", model_probabilities.max())

Number of test predictions: 6163
Minimum probability: 0.05148838088288162
Maximum probability: 0.9407631583948718


In [119]:
# Apply the Week-4 baseline rule to the test set

baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    baseline_test["days_since_last_update"].between(91, 180).astype(int)
    * (baseline_test["impressions_90d"] >= 300).astype(int)
    * baseline_test["impressions_90d"]
)

# Rank baseline recommendations from highest score to lowest
baseline_ranked = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).copy()

# Look at the top 20 baseline recommendations
baseline_top20 = baseline_ranked.head(20)

print("Baseline top 20:")
print(
    baseline_top20[
        [
            "content_id",
            "baseline_score",
            "days_since_last_update",
            "impressions_90d",
            "is_declining_label"
        ]
    ].to_string(index=False)
)

Baseline top 20:
          content_id  baseline_score  days_since_last_update  impressions_90d  is_declining_label
content_5fe46e04994d          517715                     104           517715                   1
content_9532f197bbc8          309192                     104           309192                   1
content_1aa219431528          151541                     104           151541                   0
content_3ad3a781fa91          145044                     104           145044                   0
content_e6e15ac13287          128704                     104           128704                   0
content_fac19fcdfb85          126611                     104           126611                   0
content_4a18c07e5357          125049                     104           125049                   1
content_e5ae436f9a16          117741                     104           117741                   0
content_37106924f264           89311                     104            89311                   0
con

In [120]:
# Verify that the baseline rule was applied exactly as intended

check = baseline_ranked[
    baseline_ranked["baseline_score"] > 0
][
    ["days_since_last_update", "impressions_90d", "baseline_score"]
]

print("Positive baseline scores:", len(check))
print("\nDays-since-update range among positive scores:")
print(
    check["days_since_last_update"].min(),
    "to",
    check["days_since_last_update"].max()
)

print("\nAny positive-score rows outside 91–180 days?")
print(
    (
        ~check["days_since_last_update"].between(91, 180)
    ).sum()
)

Positive baseline scores: 684

Days-since-update range among positive scores:
92 to 106

Any positive-score rows outside 91–180 days?
0


In [121]:
# Calculate Precision@20 for the baseline

baseline_top20 = baseline_ranked.head(20)

baseline_precision_at_20 = (
    baseline_top20["is_declining_label"].sum() / 20
)

print("Baseline Precision@20:", baseline_precision_at_20)
print(
    "Declining pages in baseline Top-20:",
    baseline_top20["is_declining_label"].sum()
)

Baseline Precision@20: 0.2
Declining pages in baseline Top-20: 4


In [122]:
# Rank test pages by Logistic Regression probability

model_results = test_df[
    ["content_id", "client_id", "is_declining_label"]
].copy()

model_results["model_probability"] = model_probabilities

model_ranked = model_results.sort_values(
    "model_probability",
    ascending=False
).copy()

# Select the top 20 model recommendations
model_top20 = model_ranked.head(20)

model_precision_at_20 = (
    model_top20["is_declining_label"].sum() / 20
)

print("Logistic Regression Precision@20:", model_precision_at_20)
print(
    "Declining pages in model Top-20:",
    model_top20["is_declining_label"].sum()
)

Logistic Regression Precision@20: 0.65
Declining pages in model Top-20: 13


In [123]:
# Final model-vs-baseline comparison

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "Declining in Top-20": [
        int(baseline_top20["is_declining_label"].sum()),
        int(model_top20["is_declining_label"].sum())
    ]
})

comparison

,Method,Precision@20,Declining in Top-20
0,Week-4 baseline,0.20,4
1,Logistic Regression,0.65,13


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Section 4 interpretation

The Logistic Regression model achieved a Precision@20 of 0.65, compared with 0.20 for the Week-4 baseline. This means 13 of the model's top 20 recommendations were actually labeled as declining, compared with 4 of 20 for the baseline.

The model's strongest coefficients were `users_90d`, `position_tier_top_3`, and `sessions_90d`. These are associations learned by the model and should not be interpreted as causal effects. The `users_90d` distribution is highly skewed because its mean differs substantially from its median, so its coefficient should be interpreted cautiously.

The model produced 7 false positives in its Top-20. All seven were `keyword article` content, suggesting that distinguishing declining from non-declining pages within this content type may be difficult for the current feature set.

The three examined false positives show why the model can make confident mistakes. Some non-declining pages have combinations of freshness, visibility, position, and engagement signals that resemble patterns associated with declining content.

Overall, Logistic Regression produced a substantially stronger Top-20 ranking than the simple baseline on this held-out test set, but the error analysis shows that the model is not perfect and that its high-confidence predictions should still be reviewed.

### Three concrete false-positive cases

The three highest-confidence false positives illustrate why the model can be wrong.

1. The first case received a predicted probability of about 0.92 but was not labeled declining. It had only 235 impressions, a 0.85% CTR, and an average position of 31.0. These performance signals can resemble weak content performance even though the page had only been updated 20 days earlier.

2. The second case received a predicted probability of about 0.91 but was also not labeled declining. It had 290 impressions, a 2.00% CTR, and an average position of 5.9, while also being updated 20 days earlier. Its combination of signals does not cleanly match the declining label, making it difficult for the model to classify correctly.

3. The third case received a predicted probability of about 0.90 but was not labeled declining. It had 104 days since its last update, 3,115 impressions, zero clicks, and a 0% CTR. These characteristics provide several signals that can make the item look like declining content even though its observed label is 0.

These examples show that the model's errors are not random-looking from the available features. Some non-declining pages have combinations of age, visibility, position, and engagement signals that resemble the patterns associated with declining pages.

In [124]:
# Identify the model's Top-20 mistakes

model_top20 = model_top20.copy()

model_top20["prediction_correct"] = (
    model_top20["is_declining_label"] == 1
)

model_errors = model_top20[
    model_top20["is_declining_label"] == 0
].copy()

print("Model Top-20 errors:", len(model_errors))

model_errors[
    [
        "content_id",
        "client_id",
        "model_probability",
        "is_declining_label"
    ]
]

Model Top-20 errors: 7


,content_id,client_id,model_probability,is_declining_label
10175,content_374e795aab68,client_f369cb89fc,0.920626,0
26614,content_7be5f150dc65,client_f369cb89fc,0.906939,0
20736,content_41baf0722ad9,client_8527a891e2,0.901197,0
18531,content_d10f9ce1e0cd,client_4e07408562,0.891038,0
11887,content_ce59581533ca,client_8527a891e2,0.884098,0
4905,content_f0d98be4b42c,client_4e07408562,0.883487,0
8016,content_c94a53e3bfb8,client_f369cb89fc,0.882355,0


In [125]:
# Inspect the features of the model's incorrect Top-20 recommendations

error_ids = model_errors["content_id"].tolist()

# Get the original feature values from test_df
error_details = test_df[
    test_df["content_id"].isin(error_ids)
][
    [
        "content_id",
        "client_id",
        "is_declining_label",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "content_age_days",
        "content_type"
    ]
].copy()

# Add the model probability from model_errors
error_details = error_details.merge(
    model_errors[["content_id", "model_probability"]],
    on="content_id",
    how="left"
)

# Put probability near the front
error_details = error_details[
    [
        "content_id",
        "client_id",
        "model_probability",
        "is_declining_label",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "content_age_days",
        "content_type"
    ]
].sort_values(
    "model_probability",
    ascending=False
)

error_details

,content_id,client_id,model_probability,is_declining_label,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days,content_type
2,content_374e795aab68,client_f369cb89fc,0.920626,0,20,235,2,0.85,31.0,0.00,181,keyword article
6,content_7be5f150dc65,client_f369cb89fc,0.906939,0,20,290,0,0.00,5.9,0.00,96,keyword article
5,content_41baf0722ad9,client_8527a891e2,0.901197,0,104,3115,0,0.00,12.8,0.00,275,keyword article
4,content_d10f9ce1e0cd,client_4e07408562,0.891038,0,104,166,1,0.60,16.1,0.00,348,keyword article
3,content_ce59581533ca,client_8527a891e2,0.884098,0,102,289,2,0.69,18.8,0.00,223,keyword article
0,content_f0d98be4b42c,client_4e07408562,0.883487,0,104,5818,9,0.15,5.1,22.22,230,keyword article
1,content_c94a53e3bfb8,client_f369cb89fc,0.882355,0,20,2164,5,0.23,8.1,0.00,95,keyword article


In [126]:
# Inspect Logistic Regression coefficients

# Get the fitted preprocessing and classifier
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

# Get the feature names after preprocessing
feature_names = fitted_preprocessor.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = fitted_classifier.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
)

print("Top 15 features by absolute coefficient:")
feature_importance.head(15)

Top 15 features by absolute coefficient:


,feature,coefficient,abs_coefficient
11,numeric__users_90d,-1.117210,1.117210
64,categorical__position_tier_top_3,-1.029112,1.029112
10,numeric__sessions_90d,0.998506,0.998506
15,numeric__days_with_impressions,0.771697,0.771697
33,categorical__main_intent_unknown,0.668938,0.668938
43,categorical__freshness_tier_181+,-0.651443,0.651443
16,numeric__days_with_sessions,-0.524206,0.524206
46,categorical__word_count_tier_1000-2000,0.522376,0.522376
25,categorical__competition_level_unknown,-0.518323,0.518323
31,categorical__main_intent_navigational,-0.471193,0.471193


In [127]:
# Inspect the distributions of the top 3 model features by target

top_features = [
    "users_90d",
    "position_tier",
    "sessions_90d"
]

for feature in top_features:
    print(f"\n--- {feature} ---")

    if feature == "position_tier":
        print(
            train_df.groupby(
                ["position_tier", "is_declining_label"]
            ).size().unstack(fill_value=0)
        )
    else:
        print(
            train_df.groupby("is_declining_label")[feature]
            .agg(["count", "mean", "median"])
        )


--- users_90d ---
                    count       mean  median
is_declining_label                          
0                   10724  44.821335     9.0
1                   13113  39.315107    10.0

--- position_tier ---
is_declining_label     0     1
position_tier                 
deep                 685   354
page_1              3836  5160
page_3_5            2514  3455
striking            2121  3721
top_3               1568   423

--- sessions_90d ---
                    count     mean  median
is_declining_label                        
0                   10724  46.3384     9.0
1                   13113  40.4115    10.0


In [128]:
# Select three concrete false-positive cases from the model's Top-20

three_wrong_cases = model_errors.head(3).copy()

three_wrong_cases[
    [
        "content_id",
        "client_id",
        "model_probability",
        "is_declining_label"
    ]
]

,content_id,client_id,model_probability,is_declining_label
10175,content_374e795aab68,client_f369cb89fc,0.920626,0
26614,content_7be5f150dc65,client_f369cb89fc,0.906939,0
20736,content_41baf0722ad9,client_8527a891e2,0.901197,0


In [129]:
# Check the content-type distribution of the model's Top-20 errors

print("Content type distribution among model errors:")
print(
    error_details["content_type"]
    .value_counts()
)

print("\nDeclining rate among model errors by content type:")
print(
    error_details.groupby("content_type")["is_declining_label"]
    .agg(["count", "mean"])
)

Content type distribution among model errors:
content_type
keyword article    7
Name: count, dtype: int64

Declining rate among model errors by content type:
                 count  mean
content_type                
keyword article      7   0.0


In [130]:
# Inspect the three concrete false-positive cases

three_ids = three_wrong_cases["content_id"].tolist()

three_case_details = error_details[
    error_details["content_id"].isin(three_ids)
].copy()

three_case_details

,content_id,client_id,model_probability,is_declining_label,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days,content_type
2,content_374e795aab68,client_f369cb89fc,0.920626,0,20,235,2,0.85,31.0,0.0,181,keyword article
6,content_7be5f150dc65,client_f369cb89fc,0.906939,0,20,290,0,0.00,5.9,0.0,96,keyword article
5,content_41baf0722ad9,client_8527a891e2,0.901197,0,104,3115,0,0.00,12.8,0.0,275,keyword article


In [131]:
# Final model-vs-baseline comparison

test_base_rate = y_test.mean()

comparison_final = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "Declining in Top-20": [
        int(baseline_top20["is_declining_label"].sum()),
        int(model_top20["is_declining_label"].sum())
    ],
    "Test-set base rate": [
        test_base_rate,
        test_base_rate
    ]
})

comparison_final

,Method,Precision@20,Declining in Top-20,Test-set base rate
0,Week-4 baseline,0.20,4,0.510952
1,Logistic Regression,0.65,13,0.510952


In [132]:
test_base_rate = y_test.mean()

comparison_final = pd.DataFrame({
    "Method": [
        "Test-set base rate",
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        np.nan,
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "Declining in Top-20": [
        np.nan,
        int(baseline_top20["is_declining_label"].sum()),
        int(model_top20["is_declining_label"].sum())
    ]
})

comparison_final

,Method,Precision@20,Declining in Top-20
0,Test-set base rate,NaN,NaN
1,Week-4 baseline,0.20,4.0
2,Logistic Regression,0.65,13.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Final conclusion

This experiment compared the Week-4 rule-based baseline with Logistic Regression using the same held-out test set and Precision@20 metric.

The baseline achieved a Precision@20 of 0.20, while Logistic Regression achieved 0.65. The model therefore produced a substantially more effective Top-20 review queue on this test set.

The improvement should not be interpreted as proof that the model will perform equally well on future data. The grouped client split reduces client leakage, and the error analysis shows that the model still makes confident false-positive predictions, particularly among keyword articles.

The model's strongest signals were `users_90d`, `position_tier_top_3`, and `sessions_90d`. These are predictive associations rather than causal explanations.

The experiment supports using Logistic Regression as a stronger first learned model than the simple baseline, while retaining human review of its recommendations.

In [133]:
# Self-check: verify the main honesty requirements

# 1. Reproducibility
assert RANDOM_STATE == 42

# 2. No client overlap
assert len(train_clients & test_clients) == 0

# 3. Baseline and model use the same test rows
assert set(baseline_test["content_id"]) == set(test_df["content_id"])
assert set(model_results["content_id"]) == set(test_df["content_id"])

# 4. No forbidden features were included
assert len(unexpected) == 0

# 5. Target is not included as a model feature
assert target not in selected_features

# 6. Both methods have Precision@20 results
assert not pd.isna(baseline_precision_at_20)
assert not pd.isna(model_precision_at_20)

# 7. Both methods have exactly 20 recommendations
assert len(baseline_top20) == 20
assert len(model_top20) == 20

print("SELF-CHECK PASSED")
print()
print("Random seed:", RANDOM_STATE)
print("Client overlap:", len(train_clients & test_clients))
print("Test rows:", len(test_df))
print("Forbidden features included:", unexpected)
print("Baseline Precision@20:", baseline_precision_at_20)
print("Model Precision@20:", model_precision_at_20)
print("Test-set base rate:", test_base_rate)

SELF-CHECK PASSED

Random seed: 42
Client overlap: 0
Test rows: 6163
Forbidden features included: set()
Baseline Precision@20: 0.2
Model Precision@20: 0.65
Test-set base rate: 0.5109524582184002
